# New

In [1]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
df["label"].value_counts()

label
2 (Jazz)                    1444
1 (Hip-Hop + Pop + HJDB)    1426
3 (Rock + Pop)              1269
4 (Classical)               1054
1 + 3                        162
2 + 3                        126
2 + 4                         75
Name: count, dtype: int64

In [ ]:
df_gtzan = df[df["file"].str.startswith("gtzan")]   # sanity check thtat gtzan belongs to 1 cluster
df_gtzan["label"].unique()

array(['3 (Rock + Pop)', '2 (Jazz)', '4 (Classical)',
       '1 (Hip-Hop + Pop + HJDB)'], dtype=object)

In [47]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_hard_6_2clusters_new.csv")
print(df["label"].value_counts())
df_gtzan = df[df["file"].str.startswith("gtzan")]   # sanity check thtat gtzan belongs to 1 cluster
print(df_gtzan["label"].unique())

label
1 (Pop + Groove + Rock + Hip-Hop)    2993
2 (Classical + Jazz)                 2563
Name: count, dtype: int64
['1 (Pop + Groove + Rock + Hip-Hop)' '2 (Classical + Jazz)']


In [11]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")
# some weird formatting for some files: some of them are of the form file/track, that is why we remove the second part
df["file"] = df["file"].apply(lambda x: x.split("/")[0])
rwc_files = [file for file in df["file"].values if "rwc" in file]
df["label"].value_counts()
#df.to_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")

label
3 (Jazz + Pop + Rock)              1912
1 (Classical + Groove + Jazz)      1822
2 (Pop + Rock + HJDB + Hip-Hop)    1822
Name: count, dtype: int64

In [ ]:
import numpy as np
import os
import shutil
from tqdm import tqdm
# cluster number starts with 1 here!
def prepare_cluster_data(cluster_number, clustering_configuration, SAVE_NPZ, df_path):
    root = ""
    root_save = os.path.join(root, "clustering_configurations", clustering_configuration)
    save_spectrograms_path = os.path.join(root_save,  f"cluster_{cluster_number}/data/audio/spectrograms")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")

    shutil.copytree("annotations", annotations_path, dirs_exist_ok = True)
    df = pd.read_csv(df_path)
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    print(len(df_filtered))
    print(datasets_used)

    gtzan_files = 0
    for dataset in tqdm(datasets_used):
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
        path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
        if SAVE_NPZ:
            np.savez(path_to_save, **dataset_files)
            print(f"saved npz of {dataset}")


In [ ]:
for cluster_number in range(1,4):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters3_layer113" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_raw_113_3clusters_new.csv")

In [ ]:
for cluster_number in range(1,3):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters2_layer6" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv")

In [ ]:
for cluster_number in range(1,5):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters4_layer12" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv")

## analysing the data tables

In [41]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
df["label"].value_counts()

label
2 (Jazz)                    1444
1 (Hip-Hop + Pop + HJDB)    1426
3 (Rock + Pop)              1269
4 (Classical)               1054
1 + 3                        162
2 + 3                        126
2 + 4                         75
Name: count, dtype: int64

In [110]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_hard_6_2clusters_new.csv")
df["label"].value_counts()

label
1 (Pop + Groove + Rock + Hip-Hop)    2993
2 (Classical + Jazz)                 2563
Name: count, dtype: int64

In [15]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")
df["label"].value_counts()

label
3 (Jazz + Pop + Rock)              1912
1 (Classical + Groove + Jazz)      1822
2 (Pop + Rock + HJDB + Hip-Hop)    1822
Name: count, dtype: int64

In [ ]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
root = "/beat_this/"
def get_split_files(df, cluster_number):
    
    cluster = {}
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        #print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        if dataset != "gtzan":
            split = pd.read_csv(f"/beat_this/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
    data_npz = np.load(os.path.join("/beat_this/clustering_configurations/clusters3_layer113", f"cluster_{cluster_number}", "data/audio/spectrograms", f"gtzan.npz"))
    lst = data_npz.files
    test_len = len(lst)
        
    return train_val_split, test_len, lst

In [3]:
def get_split_percentage(train_val_split):
    validation_items = 0 #4
    train_items = 0
    for keys, values in train_val_split.items():
        validation_items += values[0]
        train_items += values[1]
    print(validation_items)
    print(train_items)
    print(f"{validation_items / train_items * 100}%")
    return validation_items, train_items

In [ ]:
# 3-cluster solution
total_train = 0
total_val = 0
total_test = 0
test_files_clusters = []
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,4):
    
    train_val_split, test_len, test_files = get_split_files(df, cluster_number)
    test_files_clusters += test_files
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    total_test += test_len
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + total_test}")

100%|██████████| 16/16 [00:01<00:00,  8.76it/s]


results for the cluster 1 ( (Classical + Groove + Jazz))
184
1434
12.831241283124129%
total number of files cluster 1 is 1822 


100%|██████████| 12/12 [00:00<00:00, 25.27it/s]


results for the cluster 2 ( (Pop + Rock + HJDB + Hip-Hop))
204
1212
16.831683168316832%
total number of files cluster 2 is 1822 


100%|██████████| 15/15 [00:00<00:00, 26.67it/s]

results for the cluster 3 ( (Jazz + Pop + Rock))
168
1355
12.398523985239853%
total number of files cluster 3 is 1912 
overall results:
556
4001
13.896525868532866
total files 5556


In [ ]:
# 2-cluster solution
total_train = 0
total_val = 0
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,3):
    
    train_val_split, test_len = get_split_files(df, cluster_number)
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + 993}")

100%|██████████| 14/14 [00:00<00:00, 21.96it/s]


results for the cluster 1 ( (Pop + Groove + Rock + Hip-Hop))
347
1994
17.402206619859577%
total number of files cluster 1 is 2579 


100%|██████████| 14/14 [00:00<00:00, 27.25it/s]

results for the cluster 2 ( (Classical + Jazz))
209
2007
10.413552566018934%
total number of files cluster 2 is 2437 
overall results:
556
4001
13.896525868532866
total files 5550


In [ ]:
# 4-clsuter solution
total_train = 0
total_val = 0
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,5):
    
    train_val_split, test_len = get_split_files(df, cluster_number)
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + 993}")

100%|██████████| 14/14 [00:01<00:00,  8.18it/s]


results for the cluster 1 ( (Hip-Hop + Pop + HJDB))
220
1130
19.469026548672566%
total number of files cluster 1 is 1588 


100%|██████████| 13/13 [00:00<00:00, 23.61it/s]


results for the cluster 2 ( (Jazz))
123
1301
9.454265949269793%
total number of files cluster 2 is 1645 


100%|██████████| 11/11 [00:00<00:00, 22.37it/s]


results for the cluster 3 ( (Rock + Pop))
153
969
15.789473684210526%
total number of files cluster 3 is 1557 


100%|██████████| 15/15 [00:00<00:00, 23.85it/s]


results for the cluster 4 ( (Classical))
106
918
11.546840958605664%
total number of files cluster 4 is 1129 
overall results:
602
4318
13.941639647985179
total files 5913


# test scores

In [ ]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
def get_split_files():
    root = "/beat_this"
    datasets_used =  os.listdir(root)
    datasets_used = [file[:-4] for file in datasets_used if file.endswith("npz")]
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = npz_files_filtered  
        if dataset != "gtzan":
            split = pd.read_csv(f"/beat_this/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
        
    return train_val_split
    


In [9]:
train_val_split =  get_split_files()
train_val_split

100%|██████████| 16/16 [00:00<00:00, 17.78it/s]


{'beatles': (27, 153),
 'groove_midi': (51, 285),
 'hjdb': (35, 200),
 'guitarset': (27, 153),
 'harmonix': (137, 774),
 'rwc': (34, 192),
 'ballroom': (103, 582),
 'smc': (0, 217),
 'candombe': (5, 30),
 'jaah': (17, 96),
 'simac': (0, 595),
 'tapcorrect': (15, 86),
 'asap': (65, 408),
 'hainsworth': (33, 189),
 'filosax': (7, 41)}

In [ ]:
sum_train = 0
sum_val = 0
for key, value in train_val_split.items():
    sum_train += value[1]
    sum_val += value[0]
print(f"Total of train items : {sum_train}, total of val items : {sum_val}, total items :{sum_train + sum_val + 999}")

Total of train items : 4001, total of val items : 556, total items :5556
